In [ ]:
from abc import ABC, abstractmethod

# ---------------- Custom Exceptions ----------------
class InvalidInputError(Exception): pass
class InsufficientQuantityError(Exception): pass
class CustomerNotFoundError(Exception): pass
class EmployeeNotFoundError(Exception): pass
class ItemNotFoundError(Exception): pass

# ---------------- Abstract Base Class ----------------
class Person(ABC):
    def display_info(self): pass

# ---------------- Employee Class ----------------
class Employee(Person):
    def __init__(self, emp_id, name, role, salary):
        self.emp_id = str(emp_id).strip()
        self.name = name.strip()
        self.role = role.strip()
        self._salary = float(salary)

    def display_info(self):
        print(self)

    def __str__(self):
        return f"ID: {self.emp_id} | Name: {self.name} | Role: {self.role} | Salary: ${self._salary:,.2f}"

    def get_salary(self):
        return self._salary

# ---------------- Inventory Item ----------------
class InventoryItem:
    def __init__(self, item_id, name, quantity, price):
        self.item_id = str(item_id).strip()
        self.name = name.strip()
        self.quantity = int(quantity)
        self.price = float(price)

    def display(self):
        print(self)

    def __str__(self):
        return f"ID: {self.item_id} | Name: {self.name} | Qty: {self.quantity} | Price: ${self.price:,.2f}"

# ---------------- Customer Class ----------------
class Customer(Person):
    def __init__(self, cust_id, name):
        self.cust_id = str(cust_id).strip()
        self.name = name.strip()
        self._purchases = {}

        total_items = sum(self._purchases.values())
        print(f"ID: {self.cust_id} | Name: {self.name} | Total Items Purchased: {total_items}")

    def purchase(self, item, quantity):
        if quantity <= 0:
            raise InvalidInputError("Quantity must be positive.")
        if quantity > item.quantity:
            raise InsufficientQuantityError(f"Not enough {item.name} in stock. Available: {item.quantity}")
        
        item.quantity -= quantity
        self._purchases[item.item_id] = self._purchases.get(item.item_id, 0) + quantity
        total_cost = item.price * quantity
        print(f"{self.name} purchased {quantity} of {item.name} for ${total_cost:,.2f}")

    def __call__(self):
        print(f"\n{self.name}'s Purchase Summary:")
        if not self._purchases:
            print("No purchases yet.")
            return
        for item_id, qty in self._purchases.items():
            print(f"Item ID: {item_id} | Quantity: {qty}")

# ---------------- Management System ----------------
class ManagementSystem:
    def __init__(self):
        self.employees = {}
        self.inventory = {}
        self.customers = {}

    def add_employee(self, emp):
        if emp.emp_id in self.employees:
            raise InvalidInputError("Employee with this ID already exists!")
        self.employees[emp.emp_id] = emp
        print("Employee added successfully!")

    def display_employees(self):
        if not self.employees:
            print("No employees found!")
            return
        print("\n--- Employees ---")
        for emp in self.employees.values():
            emp.display_info()

    def highest_paid_employee(self):
        if not self.employees:
            raise EmployeeNotFoundError("No employees found!")
        highest = max(self.employees.values(), key=lambda e: e.get_salary())
        print("\n--- Highest Paid Employee ---")
        highest.display_info()
        return highest

    def add_inventory_item(self, item):
        if item.item_id in self.inventory:
            raise InvalidInputError("Item with this ID already exists!")
        self.inventory[item.item_id] = item
        print("Inventory item added successfully!")

    def display_inventory(self):
        if not self.inventory:
            print("No inventory items found!")
            return
        print("\n--- Inventory ---")
        for item in self.inventory.values():
            item.display()

    def get_low_stock_items(self, threshold=5):
        low_stock = [item for item in self.inventory.values() if item.quantity < threshold]
        if low_stock:
            print(f"\n--- Low Stock Items (below {threshold}) ---")
            for item in low_stock:
                item.display()
        return low_stock

    def add_customer(self, cust):
        if cust.cust_id in self.customers:
            raise InvalidInputError("Customer with this ID already exists!")
        self.customers[cust.cust_id] = cust
        print("Customer added successfully!")

    def display_customers(self):
        if not self.customers:
            print("No customers found!")
            return
        print("\n--- Customers ---")
        for cust in self.customers.values():
            cust.display_info()

    def dashboard(self):
        print("\n=== SYSTEM DASHBOARD ===")
        print(f"Total Employees: {len(self.employees)}")
        print(f"Total Inventory Items: {len(self.inventory)}")
        print(f"Total Customers: {len(self.customers)}")
        total_stock_value = sum(item.price * item.quantity for item in self.inventory.values())
        print(f"Total Stock Value: ${total_stock_value:,.2f}")

# ---------------- Interactive Console ----------------
def main():
    system = ManagementSystem()

    while True:
        print("\n" + "="*50)
        print("          MANAGEMENT SYSTEM MENU")
        print("="*50)
        print("1. Add Employee")
        print("2. Add Inventory Item")
        print("3. Add Customer")
        print("4. Display Employees")
        print("5. Display Inventory")
        print("6. Display Customers")
        print("7. Purchase Item")
        print("8. Highest Paid Employee")
        print("9. Dashboard")
        print("10. Low Stock Alert")
        print("11. Exit")
        print("="*50)

        choice = input("Enter your choice (1-11): ").strip()

        try:
            if choice == "1":
                emp_id = input("Employee ID: ").strip()
                name = input("Name: ").strip()
                role = input("Role: ").strip()
                salary = float(input("Salary: $"))
                system.add_employee(Employee(emp_id, name, role, salary))

            elif choice == "2":
                item_id = input("Item ID: ").strip()
                name = input("Item Name: ").strip()
                quantity = int(input("Quantity: "))
                price = float(input("Price: $"))
                system.add_inventory_item(InventoryItem(item_id, name, quantity, price))

            elif choice == "3":
                cust_id = input("Customer ID: ").strip()
                name = input("Customer Name: ").strip()
                system.add_customer(Customer(cust_id, name))

            elif choice == "4":
                system.display_employees()

            elif choice == "5":
                system.display_inventory()

            elif choice == "6":
                system.display_customers()

            elif choice == "7":
                if not system.customers:
                    print("No customers available!")
                    continue
                if not system.inventory:
                    print("No inventory items available!")
                    continue

                system.display_customers()
                cust_id = input("\nCustomer ID: ").strip()
                
                # FIXED: Use a more robust way to find the customer
                customer = None
                for cust_key, cust_obj in system.customers.items():
                    if cust_key == cust_id:
                        customer = cust_obj
                        break
                
                if not customer:
                    print(f"DEBUG: Available Customer IDs: {list(system.customers.keys())}")
                    print(f"DEBUG: Looking for: '{cust_id}'")
                    raise CustomerNotFoundError(f"Customer with ID '{cust_id}' not found!")

                system.display_inventory()
                item_id = input("\nItem ID to purchase: ").strip()
                
                # FIXED: Use a more robust way to find the item
                item = None
                for item_key, item_obj in system.inventory.items():
                    if item_key == item_id:
                        item = item_obj
                        break
                
                if not item:
                    print(f"DEBUG: Available Item IDs: {list(system.inventory.keys())}")
                    print(f"DEBUG: Looking for: '{item_id}'")
                    raise ItemNotFoundError(f"Item with ID '{item_id}' not found!")

                quantity = int(input("Quantity to purchase: "))
                customer.purchase(item, quantity)
                customer()

            elif choice == "8":
                system.highest_paid_employee()

            elif choice == "9":
                system.dashboard()

            elif choice == "10":
                system.get_low_stock_items()

            elif choice == "11":
                print("Thank you for using the Management System. Goodbye!")
                break

            else:
                raise InvalidInputError("Please choose a valid option (1-11)")

        except (ValueError, InvalidInputError, InsufficientQuantityError, 
                CustomerNotFoundError, EmployeeNotFoundError, ItemNotFoundError) as e:
            print(f" Error: {e}")
        except Exception as e:
            print(f" Unexpected Error: {e}")

if __name__ == "__main__":
    main()


          MANAGEMENT SYSTEM MENU
1. Add Employee
2. Add Inventory Item
3. Add Customer
4. Display Employees
5. Display Inventory
6. Display Customers
7. Purchase Item
8. Highest Paid Employee
9. Dashboard
10. Low Stock Alert
11. Exit


Enter your choice (1-11):  1
Employee ID:  1
Name:  Asim Ali
Role:  Developer
Salary: $ 30000


Employee added successfully!

          MANAGEMENT SYSTEM MENU
1. Add Employee
2. Add Inventory Item
3. Add Customer
4. Display Employees
5. Display Inventory
6. Display Customers
7. Purchase Item
8. Highest Paid Employee
9. Dashboard
10. Low Stock Alert
11. Exit


Enter your choice (1-11):  4



--- Employees ---
ID: 1 | Name: Asim Ali | Role: Developer | Salary: $30,000.00

          MANAGEMENT SYSTEM MENU
1. Add Employee
2. Add Inventory Item
3. Add Customer
4. Display Employees
5. Display Inventory
6. Display Customers
7. Purchase Item
8. Highest Paid Employee
9. Dashboard
10. Low Stock Alert
11. Exit


Enter your choice (1-11):  1
Employee ID:  2
Name:  Hashim Ali
Role:  Manager
Salary: $ 50000


... (33 outputs left out)

Enter your choice (1-11):  2
Item ID:  5
Item Name:  Earpod
Quantity:  4
Price: $ 7


Inventory item added successfully!

          MANAGEMENT SYSTEM MENU
1. Add Employee
2. Add Inventory Item
3. Add Customer
4. Display Employees
5. Display Inventory
6. Display Customers
7. Purchase Item
8. Highest Paid Employee
9. Dashboard
10. Low Stock Alert
11. Exit


Enter your choice (1-11):  10



--- Low Stock Items (below 5) ---
ID: 5 | Name: Earpod | Qty: 4 | Price: $7.00

          MANAGEMENT SYSTEM MENU
1. Add Employee
2. Add Inventory Item
3. Add Customer
4. Display Employees
5. Display Inventory
6. Display Customers
7. Purchase Item
8. Highest Paid Employee
9. Dashboard
10. Low Stock Alert
11. Exit
